# Part 4: Baseline, RAG, and Fine-Tuning Comparison

Team 3 CozyDay · Ashley Zhou

This final comparison uses the same five questions, the same original `slm_exp5.pt` checkpoint lineage, seed 42, temperature 1.0, and 30 generated characters. It reads the committed results from Lin's RAG experiment and Shivani's fine-tuning experiment rather than rerunning or modifying either model.

For the RAG condition, Top-1 is used. All four tested Top-K settings received the same mean quality score (1.0/10) and produced 0/5 correct answers. Top-1 is therefore the simplest tied setting, retrieves the correct document for every question, and places the least pressure on the SLM's 32-character context window.

In [1]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()
DATA_DIR = next((d for d in (cwd, cwd / 'a7_part2') if (d / 'part2_quality_evaluation.csv').is_file()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Run from the repository root or a7_part2/.')

rag_all = pd.read_csv(DATA_DIR / 'part2_quality_evaluation.csv', keep_default_na=False)
fine = pd.read_csv(DATA_DIR / 'part3_finetune_results.csv', keep_default_na=False)
probe = pd.read_csv(DATA_DIR / 'part3_unseen_probe.csv', keep_default_na=False)

baseline = rag_all[rag_all['method'] == 'No-RAG matched-format'].copy()
rag_rows = rag_all[rag_all['method'].str.endswith('RAG revised')].copy()
rag_top1 = rag_all[rag_all['method'] == 'Top-1 RAG revised'].copy()

expected_ids = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5']
for frame in (baseline, rag_top1, fine):
    assert frame['question_id'].tolist() == expected_ids
assert baseline['question'].tolist() == rag_top1['question'].tolist() == fine['question'].tolist()
assert baseline['expected_answer'].tolist() == rag_top1['expected_answer'].tolist() == fine['expected_answer'].tolist()
assert set(fine['seed'].astype(int)) == {42}
assert set(fine['temperature'].astype(float)) == {1.0}
assert set(fine['max_new_tokens'].astype(int)) == {30}
assert set(rag_top1['seed'].astype(float).astype(int)) == {42}
assert set(rag_top1['temperature'].astype(float)) == {1.0}
assert set(rag_top1['max_new_tokens'].astype(float).astype(int)) == {30}

rag_rows['answer_found_bool'] = rag_rows['answer_found'].astype(str).str.lower().eq('true')
topk_summary = (rag_rows.groupby('top_k', as_index=False)
                .agg(mean_quality=('quality_score', 'mean'),
                     correct_answers=('answer_found_bool', 'sum'),
                     questions=('question_id', 'count'))
                .sort_values('top_k'))
assert len(topk_summary) == 4
assert (topk_summary['mean_quality'] == 1.0).all()
assert (topk_summary['correct_answers'] == 0).all()
display(topk_summary)


,top_k,mean_quality,correct_answers,questions
0,1,1.0,0,5
1,2,1.0,0,5
2,3,1.0,0,5
3,5,1.0,0,5


## Three-condition comparison

The table below preserves the generated text exactly, including line breaks and incomplete fragments.

In [2]:
comparison = pd.DataFrame({
    'question_id': expected_ids,
    'question': baseline['question'].tolist(),
    'expected_answer': baseline['expected_answer'].tolist(),
    'baseline_slm': baseline['response'].tolist(),
    'slm_plus_rag_top1': rag_top1['response'].tolist(),
    'fine_tuned_slm': fine['after_response'].tolist(),
})
comparison['baseline_correct'] = baseline['answer_found'].astype(str).str.lower().eq('true').tolist()
comparison['rag_correct'] = rag_top1['answer_found'].astype(str).str.lower().eq('true').tolist()
comparison['fine_tuned_correct'] = fine['after_first_line_exact'].astype(str).str.lower().eq('true').tolist()

assert comparison['baseline_correct'].sum() == 0
assert comparison['rag_correct'].sum() == 0
assert comparison['fine_tuned_correct'].sum() == 5
display(comparison[['question_id', 'question', 'baseline_slm', 'slm_plus_rag_top1', 'fine_tuned_slm']])


,question_id,question,baseline_slm,slm_plus_rag_top1,fine_tuned_slm
0,Q1,what does mia like,intelligence.\nstudents create,intelligence.\nstudents create,apples\nartim ing\npredic\nsmalli
1,Q2,what does sam like,systems.\nlanguage models can r,s useful for machine learning,music\nsam likes c\nartike apple
2,Q3,what does tom like,s.\nattention helps models use,s can run on local computers.\n,swimming\ns ap lic\ndjango apple
3,Q4,what does anna like,intelligence.\nstudents create,learning models contain parame,painting\np apneural neetworkad
4,Q5,what does lily like,helps models use contextual in,sfformaties.\nattention helps m,reading\nsnear music\ndjango app


## Condition-level evaluation

A response counts as factually accurate only when it supplies the expected answer. Relevance is judged by whether the first line directly answers the question. Unsupported-information counts include unrelated corpus fragments, garbled continuations, or facts not supported by the question's evidence.

In [3]:
condition_evaluation = pd.DataFrame([
    {
        'condition': 'Baseline SLM',
        'factual_accuracy': '0/5',
        'relevance': '0/5 directly answer the question',
        'response_consistency': 'Low: deterministic but incomplete corpus fragments; Q1 and Q4 are identical',
        'unsupported_or_invented_information': '5/5 contain unrelated corpus text',
    },
    {
        'condition': 'SLM + Top-1 RAG',
        'factual_accuracy': '0/5',
        'relevance': '0/5 directly answer the question',
        'response_consistency': 'Low: correct documents retrieved, but all outputs remain unrelated fragments',
        'unsupported_or_invented_information': '5/5 fail to use the retrieved fact',
    },
    {
        'condition': 'Fine-Tuned SLM',
        'factual_accuracy': '5/5 first lines exactly match the expected answers',
        'relevance': '5/5 begin with a direct answer',
        'response_consistency': 'High for the answer-first format; low for the remaining generated text',
        'unsupported_or_invented_information': '5/5 have unsupported tails; 3/3 unseen-person probes invent a trained fact',
    },
])
assert int(probe['unsupported_answer_given'].astype(str).str.lower().eq('true').sum()) == 3
display(condition_evaluation)

comparison.to_csv(DATA_DIR / 'part4_three_condition_comparison.csv', index=False)
condition_evaluation.to_csv(DATA_DIR / 'part4_condition_evaluation.csv', index=False)
topk_summary.to_csv(DATA_DIR / 'part4_topk_selection.csv', index=False)
print('Saved the three Part 4 CSV files in', DATA_DIR)


,condition,factual_accuracy,relevance,response_consistency,unsupported_or_invented_information
0,Baseline SLM,0/5,0/5 directly answer the question,Low: deterministic but incomplete corpus fragm...,5/5 contain unrelated corpus text
1,SLM + Top-1 RAG,0/5,0/5 directly answer the question,"Low: correct documents retrieved, but all outp...",5/5 fail to use the retrieved fact
2,Fine-Tuned SLM,5/5 first lines exactly match the expected ans...,5/5 begin with a direct answer,High for the answer-first format; low for the ...,5/5 have unsupported tails; 3/3 unseen-person ...


Saved the three Part 4 CSV files in /Users/qingqing/Documents/Codex/2026-09-05/https-docs-google-com-document-d/cozyday-merge/a7_part2


## Part 4 final analysis

**1. Which condition produced the most accurate responses?** The fine-tuned SLM was the most accurate: its first line exactly matched the expected answer for 5/5 questions, compared with 0/5 for both the baseline and Top-1 RAG. This result shows recall of the training examples, not generalization, because the five evaluation questions appeared in the fine-tuning data.

**2. Which approach required updating model weights?** Only fine-tuning updated the weights: all 30 parameter tensors changed and their L2 distance from the original weights was 9.5424. Baseline generation and RAG used the unchanged original checkpoint; RAG changed only the information supplied in the prompt.

**3. Which approach would be easier to maintain when information changes?** RAG would be easier to maintain because a changed fact can be replaced in the knowledge base without retraining the SLM. Fine-tuning would require new examples, another training run, and regression checks such as the original-corpus validation loss, which rose from 0.1240 to 0.8347 in this experiment.

**4. Which approach would you recommend for a startup with frequently changing policies?** I would recommend RAG for frequently changing policies because it separates current policy content from model weights and makes updates auditable. However, this experiment's tiny character-level SLM scored only 1.0/10 and 0/5 correct under every Top-K setting, so a production system would also need a more capable model or a larger context window.

**5. Could RAG and fine-tuning be combined in the same language model?** Yes. Fine-tuning can teach stable behavior, terminology, or response format, while RAG supplies current facts at inference time; the retrieved evidence should still be validated because fine-tuning did not eliminate unsupported output (3/3 unseen-person probes invented a trained fact).

### Overall finding

Fine-tuning clearly improved recall on the five trained questions, but it also caused forgetting and unsupported continuations. RAG preserved the original weights and would be easier to update, yet this SLM's 32-character context and limited character-level capability prevented it from using even correctly retrieved facts. The experiment therefore demonstrates the intended trade-off: changing weights can store facts but risks memorization and forgetting, while retrieval is maintainable but depends on the generator's ability to use context.